# Занятие 4. Формирование признаков и выбор метода

Прогноз оттока «СвязьРегион». Ноутбук строит на каждую дату расчёта T исторически корректный снимок «абонент × T»: активную базу, признаки только из сведений, известных на T, и метку из будущего горизонта. Затем готовит преобразование данных для первого кандидата (fit только на train) и воспроизводит базовое правило «сортировка по долгу» на train.

**Запуск** из корня репозитория, одной командой:

```
jupyter nbconvert --to notebook --execute --inplace lesson4/features.ipynb
```

Ядро работает в папке `lesson4/`, все пути ниже относительны неё. Результаты — в `lesson4/output/`, паспорт признаков — в `lesson4/docs/feature_catalog.csv`. Конфигурация дат, окна и горизонта — `lesson4/config.json`.

| Параметр | Правило |
|---|---|
| Единица наблюдения | пара `(customer_id, snapshot_at)` |
| Дата расчёта T | 00:00 по Москве первого числа месяца |
| Активная база | `activation < T` и (расторжения нет или `termination ≥ T`) — как в уставе и на этапе 3 |
| Горизонт метки | календарный месяц `[T, 1-е число следующего месяца)` |
| Окно истории | три календарных месяца до T: для T = 01.07.2026 — апрель, май, июнь |
| Доступность | каждая использованная строка любого источника: `available_at ≤ T`; принадлежности прошлому месяцу недостаточно |
| Разбиение | train 01.12.2025–01.07.2026, validation 01.08.2026, test 01.09.2026; только по времени |

Источники: очищенные таблицы этапа 2 (`lesson2/output/`: справочник, журнал событий, биллинг, поддержка) и новый `data/digital_usage_monthly.csv`. Сырой журнал этапа 1 нужен только для независимого пересчёта базы.

| Шаг | Что делает | Результат |
|---|---|---|
| 0 | Конфигурация, загрузка, проверки входа и нового цифрового источника | — |
| 1 | Активная база и метка | `output/labels.csv` |
| 2 | Признаки договора и биллинга | — |
| 3 | Признаки поддержки | — |
| 4 | Признаки цифрового источника | — |
| 5 | Сборка слоёв и паспорт признаков | `features.csv`, `split.csv`, `feature_audit.csv`, `docs/feature_catalog.csv` |
| 6 | Проверки времени и полноты | `output/checks.csv` |
| 7 | Преобразование данных для кандидата 1 | `x_columns.csv`, `matrix_shapes.csv` |
| 8 | Базовое правило на train | `baseline_train_metrics.csv` |
| 9 | Воспроизводимость | `checksums.csv` |

---

## Шаг 0. Конфигурация и загрузка

In [1]:
import hashlib
import importlib.util
import json
import math
import sys
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

if not Path("features.ipynb").exists():
    raise RuntimeError(f"Kernel must run in lesson4/, current folder is {Path.cwd()}")

# Inputs
LESSON2_OUTPUT = Path("../lesson2/output")  # cleaned tables of lesson 2
LESSON1_RAW = Path("../lesson1/data/raw")   # raw event log, only for the independent base recount
NEW_DATA = Path("data")                     # lesson 4 source
RUN_CONFIG = Path("config.json")
PREPROCESSING_TEMPLATE = Path("starter/preprocessing_template.py")
CATALOG_TEMPLATE = Path("feature_catalog_template.csv")

# Outputs
OUTPUT = Path("output")
DOCS = Path("docs")

MSK = "+03:00"
print(f"Python {sys.version.split()[0]}; pandas {pd.__version__}; numpy {np.__version__}; scikit-learn {sklearn.__version__}")

Python 3.13.17; pandas 2.3.3; numpy 2.5.4; scikit-learn 1.9.1


In [2]:
def require(condition: bool, message: str) -> None:
    """Fail fast on a violated precondition."""
    if not condition:
        raise ValueError(message)


def read_csv(path: Path) -> pd.DataFrame:
    """Read a project CSV as text: UTF-8 with BOM, ';' separator, empty cell = missing."""
    if not path.exists():
        raise FileNotFoundError(f"Input not found: {path.resolve()}")
    return pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)


def as_number(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.replace("", np.nan), errors="raise").astype(float)


def as_int(s: pd.Series) -> pd.Series:
    values = as_number(s)
    require(not (values.dropna() % 1 != 0).any(), f"{s.name}: non-integer values")
    return values.astype("Int64")


def as_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s.replace("", np.nan), format="%Y-%m-%d", errors="raise")


def as_moment(s: pd.Series) -> pd.Series:
    """ISO 8601 moments; every filled value must carry the Moscow offset."""
    filled = s[s != ""]
    foreign = filled[~filled.str.endswith(MSK)]
    require(foreign.empty, f"{s.name}: {len(foreign)} values without {MSK}")
    return pd.to_datetime(s.replace("", np.nan), format="ISO8601", errors="raise")


def as_bool(s: pd.Series) -> pd.Series:
    unexpected = set(s) - {"True", "False"}
    require(not unexpected, f"{s.name}: unexpected values {sorted(unexpected)}")
    return s == "True"


def snapshot_moment(T: pd.Timestamp) -> pd.Timestamp:
    """Calculation moment of the calendar date T: 00:00 Moscow time."""
    return pd.Timestamp(f"{T:%Y-%m-%d}T00:00:00{MSK}")


def first_of_month(text: str) -> pd.Timestamp:
    value = pd.Timestamp(text)
    require(value.day == 1 and value == value.normalize(), f"config: {text} is not the first day of a month")
    return value


def _format_float(v) -> str:
    """Fixed notation, at most 10 decimals, no float noise; '-0' is written as '0'."""
    if pd.isna(v):
        return ""
    text = f"{float(v) + 0.0:.10f}".rstrip("0").rstrip(".")
    return "0" if text in ("-0", "") else text


def write_csv(df: pd.DataFrame, path: Path) -> None:
    """Project CSV format: dates as YYYY-MM-DD, moments as ISO 8601 with offset, '\n' line ends on every OS."""
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if isinstance(s.dtype, pd.DatetimeTZDtype):
            out[col] = s.map(lambda v: v.isoformat() if pd.notna(v) else "")
        elif pd.api.types.is_datetime64_dtype(s):
            out[col] = s.dt.strftime("%Y-%m-%d").fillna("")
        elif pd.api.types.is_float_dtype(s):
            out[col] = s.map(_format_float)
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path, sep=";", encoding="utf-8-sig", index=False, lineterminator="\n")


CHECKS = []


def record_check(check: str, passed: bool, detail: str) -> None:
    """Every check is kept for output/checks.csv; a failed one stops the notebook."""
    CHECKS.append({"check": check, "passed": bool(passed), "detail": detail})
    if not passed:
        raise AssertionError(f"{check}: {detail}")

Конфигурация читается из `config.json` и проверяется. Реализованы только согласованные варианты: календарный горизонт и база «до событий на T». Любое другое значение останавливает ноутбук, а не игнорируется молча. Доля кампании хранится точной дробью, чтобы `floor(0,10 × N)` не зависел от округления float.

In [3]:
config = json.loads(RUN_CONFIG.read_text(encoding="utf-8"))
require(config["horizon"] == "calendar_month", f"horizon {config['horizon']!r} is not implemented")
require(config["active_mode"] == "before_events", f"active_mode {config['active_mode']!r} is not implemented")
require(isinstance(config["window_months"], int) and config["window_months"] >= 2,
        "window_months must be an integer >= 2: a change between two months needs two months")

WINDOW = config["window_months"]
TRAIN_DATES = pd.date_range(first_of_month(config["train_start"]), first_of_month(config["train_end_inclusive"]),
                            freq="MS")
VALIDATION_DATES = pd.DatetimeIndex([first_of_month(d) for d in config["validation_dates"]])
TEST_DATES = pd.DatetimeIndex([first_of_month(d) for d in config["test_dates"]])
ALL_DATES = TRAIN_DATES.append(VALIDATION_DATES).append(TEST_DATES)
require(len(TRAIN_DATES) and len(VALIDATION_DATES) and len(TEST_DATES), "every split part needs a date")
require(list(ALL_DATES) == sorted(set(ALL_DATES)), "train < validation < test must increase without overlap")
OUTCOMES_OBSERVED_BEFORE = first_of_month(config["observation_end_exclusive"])
CAMPAIGN_FRACTION = Fraction(str(config["campaign_fraction"]))
require(0 < CAMPAIGN_FRACTION < 1, "campaign_fraction must be in (0, 1)")

SPLIT_OF = {T: part for part, dates in (("train", TRAIN_DATES), ("validation", VALIDATION_DATES),
                                        ("test", TEST_DATES)) for T in dates}


def window_months(T: pd.Timestamp) -> list[pd.Timestamp]:
    """Calendar months of the history window, oldest first; the last one is T-1."""
    return [T - pd.DateOffset(months=k) for k in range(WINDOW, 0, -1)]


print(json.dumps(config, ensure_ascii=False, indent=2))
print("Window for T=2026-07-01:", [f"{m:%Y-%m}" for m in window_months(pd.Timestamp("2026-07-01"))])

{
  "horizon": "calendar_month",
  "active_mode": "before_events",
  "window_months": 3,
  "train_start": "2025-12-01",
  "train_end_inclusive": "2026-07-01",
  "validation_dates": [
    "2026-08-01"
  ],
  "test_dates": [
    "2026-09-01"
  ],
  "observation_end_exclusive": "2026-10-01",
  "campaign_fraction": 0.1
}
Window for T=2026-07-01: ['2026-04', '2026-05', '2026-06']


In [4]:
subscribers = read_csv(LESSON2_OUTPUT / "subscribers_clean.csv")
events = read_csv(LESSON2_OUTPUT / "contract_events_clean.csv")
billing = read_csv(LESSON2_OUTPUT / "billing_clean.csv")
support = read_csv(LESSON2_OUTPUT / "support_clean.csv")
digital = read_csv(NEW_DATA / "digital_usage_monthly.csv")
raw_events = read_csv(LESSON1_RAW / "contract_events.csv")

subscribers["activation_date"] = as_date(subscribers.activation_date)
subscribers["activation_date_restored"] = as_bool(subscribers.activation_date_restored)
for table in (events, raw_events):
    table["event_date"] = as_date(table.event_date)

billing["billing_month"] = as_date(billing.billing_month)
billing["available_at"] = as_moment(billing.available_at)
for col in ("monthly_fee_rub", "charged_rub", "paid_rub", "opening_debt_rub", "closing_debt_rub", "traffic_gb"):
    billing[col] = as_number(billing[col])
for col in ("voice_minutes", "active_days", "payment_delay_days", "tariff_changes_count"):
    billing[col] = as_int(billing[col])
for col in ("closing_debt_restored", "traffic_gb_missing", "voice_minutes_invalid"):
    billing[col] = as_bool(billing[col])

for col in ("created_at", "available_at", "resolved_at", "resolution_available_at"):
    support[col] = as_moment(support[col])
support["satisfaction_score"] = as_int(support.satisfaction_score)

DIGITAL_COUNTS = ["observed_days", "app_sessions", "payment_attempts", "payment_failures", "selfservice_actions"]
digital["usage_month"] = as_date(digital.usage_month)
digital["available_at"] = as_moment(digital.available_at)
for col in DIGITAL_COUNTS:
    digital[col] = as_int(digital[col])

pd.DataFrame({
    "table": ["subscribers_clean", "contract_events_clean", "billing_clean", "support_clean",
              "digital_usage_monthly", "lesson1 contract_events (raw)"],
    "rows": [len(subscribers), len(events), len(billing), len(support), len(digital), len(raw_events)],
    "columns": [t.shape[1] for t in (subscribers, events, billing, support, digital, raw_events)],
})

,table,rows,columns
0,subscribers_clean,2600,6
1,contract_events_clean,3299,4
2,billing_clean,24098,17
3,support_clean,10818,12
4,digital_usage_monthly,24098,9
5,lesson1 contract_events (raw),3299,4


### Проверки входа

Инварианты, на которые опираются формулы признаков. Нарушение любого останавливает ноутбук: признаки на сломанном входе бессмысленны.

In [5]:
# Keys and references
require(subscribers.customer_id.is_unique, "subscribers_clean: duplicate customer_id")
require(not events.duplicated(["customer_id", "event_type"]).any(), "more than one event of a type per customer")
require(set(events.customer_id) == set(subscribers.customer_id), "events and subscribers cover different customers")
require(not billing.duplicated(["customer_id", "billing_month"]).any(), "billing_clean: duplicate key")
require(set(billing.customer_id) <= set(subscribers.customer_id), "billing_clean: unknown customer_id")
require(support.ticket_id.is_unique, "support_clean: duplicate ticket_id")
require(set(support.customer_id) <= set(subscribers.customer_id), "support_clean: unknown customer_id")

# Billing: the formulas below assume these
require((billing.billing_month.dt.day == 1).all(), "billing_month is not a first day")
billing_month_end = (billing.billing_month + pd.DateOffset(months=1)).map(snapshot_moment)
require((billing.available_at >= billing_month_end).all(), "billing row published before its month ended")
require(billing.closing_debt_rub.notna().all() and (billing.closing_debt_rub >= 0).all(),
        "closing debt must be known and non-negative (restored on stage 2)")
require((billing.charged_rub > 0).all() and billing.paid_rub.notna().all(), "charged must be > 0, paid known")
require(billing.active_days.between(1, 31).all(), "active_days outside 1..31")
require(billing.traffic_gb.isna().equals(billing.traffic_gb_missing), "traffic_gb_missing disagrees with traffic_gb")
require(set(billing.tariff) == {"Basic", "Comfort", "Premium"}, "unexpected tariff values")

# Support: availability is never earlier than creation
require((support.available_at >= support.created_at).all(), "support: available_at before created_at")
require(set(support.topic) <= {"network", "billing", "tariff", "services", "unknown"}, "support: unexpected topic")

print("Input checks passed.")

Input checks passed.


### Новый цифровой источник

Проверки шага 3 задания: составной ключ, ссылки на абонентов, статусы, диапазоны и порядок дат. Массовой очистки источник не требует, поэтому проверки только фиксируют свойства, на которые опираются признаки. Результаты входят в `checks.csv`.

Статус и показатели — одна строка с одним `available_at`. До этого момента неизвестно ни значение, ни сам статус.

In [6]:
STATUSES = {"observed", "not_enabled", "export_gap"}
observed = digital.observation_status == "observed"
month_end = (digital.usage_month + pd.DateOffset(months=1)).map(snapshot_moment)
days_in_month = digital.usage_month.dt.days_in_month
publication_delay_h = (digital.available_at - month_end).dt.total_seconds() / 3600

dups = int(digital.duplicated(["customer_id", "usage_month"]).sum())
record_check("digital: key (customer_id, usage_month) is unique", dups == 0, f"{len(digital)} rows, duplicates: {dups}")
unknown = set(digital.customer_id) - set(subscribers.customer_id)
record_check("digital: every customer_id is in subscribers_clean", not unknown, f"unknown ids: {len(unknown)}")
pairs_equal = (set(zip(digital.customer_id, digital.usage_month))
               == set(zip(billing.customer_id, billing.billing_month)))
record_check("digital: (customer, month) pairs equal the billing pairs", pairs_equal,
             "no new customers and no new months")
record_check("digital: observation_status in {observed, not_enabled, export_gap}",
             set(digital.observation_status) <= STATUSES, str(digital.observation_status.value_counts().to_dict()))
record_check("digital: usage_month is a first day, Sep 2025 – Aug 2026",
             bool((digital.usage_month.dt.day == 1).all()
                  and digital.usage_month.between("2025-09-01", "2026-08-01").all()),
             f"{digital.usage_month.min():%Y-%m} … {digital.usage_month.max():%Y-%m}")
record_check("digital: available_at is not earlier than the end of usage_month", bool((publication_delay_h >= 0).all()),
             f"delay after month end, hours: {sorted(publication_delay_h.unique().tolist())}")

obs = digital[observed]
ranges_ok = (obs[DIGITAL_COUNTS].notna().all().all()
             and obs.observed_days.between(1, days_in_month[observed]).all()
             and (obs[["app_sessions", "payment_attempts", "payment_failures", "selfservice_actions"]] >= 0).all().all()
             and (obs.payment_failures <= obs.payment_attempts).all())
record_check("digital: observed rows have all counts, 1 <= observed_days <= month length, 0 <= failures <= attempts",
             bool(ranges_ok), f"{len(obs)} observed rows")
record_check("digital: not_enabled and export_gap rows have every count empty",
             bool(digital.loc[~observed, DIGITAL_COUNTS].isna().all().all()), f"{int((~observed).sum())} rows")

active_days = digital.merge(billing[["customer_id", "billing_month", "active_days"]],
                            left_on=["customer_id", "usage_month"], right_on=["customer_id", "billing_month"],
                            how="left", validate="1:1").active_days
same_days = (digital.observed_days[observed] == active_days[observed]).all()
record_check("digital: observed_days equals billing active_days of the same month", bool(same_days),
             "normalisation by observed_days covers exactly the active contract days")

pd.crosstab(digital.observation_status, publication_delay_h.rename("delay_h"), margins=True)

delay_h,0.0,6.0,24.0,36.0,All
observation_status,,,,,
export_gap,491,181,162,153,987
not_enabled,3039,1026,975,1080,6120
observed,8484,2902,2735,2870,16991
All,12014,4109,3872,4103,24098


Половина отчётов публикуется через 6–36 часов после конца месяца. На 00:00 первого числа такой отчёт за предыдущий месяц ещё неизвестен — вместе со статусом. Это не дефект источника, а свойство его публикации, и признаки его учитывают (шаг 4).

*Шаг 0. Конфигурация и загрузка*

---

## Шаг 1. Активная база и метка

Функции те же, что на этапе 3. Активная база строится только по журналу событий, метка — отдельной функцией; дата расторжения из неё не выходит и в признаки не попадает. Если конец горизонта позже `observation_end_exclusive`, метка неизвестна, а не 0. На всех 10 датах конфигурации горизонт наблюдается полностью.

In [7]:
def build_lifecycle(events: pd.DataFrame) -> pd.DataFrame:
    """One row per customer: activation date and optional termination date from the event log."""
    wide = events.pivot(index="customer_id", columns="event_type", values="event_date")  # raises on duplicates
    if "termination" not in wide:
        wide["termination"] = pd.NaT
    lifecycle = (wide.rename(columns={"activation": "activation_date", "termination": "termination_date"})
                 .rename_axis(columns=None).reset_index())
    require(lifecycle.activation_date.notna().all(), "customer without an activation event")
    require(not (lifecycle.termination_date < lifecycle.activation_date).any(), "termination before activation")
    return lifecycle[["customer_id", "activation_date", "termination_date"]]


def active_base(lifecycle: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """Active at T, state before the events dated T: activation < T and (no termination or termination >= T)."""
    term = lifecycle.termination_date
    active = (lifecycle.activation_date < T) & (term.isna() | (term >= T))
    return lifecycle.loc[active, ["customer_id"]].sort_values("customer_id").reset_index(drop=True)


def churn_target(base: pd.DataFrame, lifecycle: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """Future label for the base at T. The termination date never leaves this function."""
    end = T + pd.DateOffset(months=1)
    term = base.merge(lifecycle, on="customer_id", how="left", validate="1:1").termination_date
    if end > OUTCOMES_OBSERVED_BEFORE:
        target = pd.Series(pd.NA, index=base.index, dtype="Int64")  # outcome not fully observed: unknown, not 0
    else:
        target = ((term >= T) & (term < end)).astype("Int64")
    return pd.DataFrame({"customer_id": base.customer_id, "snapshot_at": snapshot_moment(T),
                         "target_end_exclusive": snapshot_moment(end), "churn_target": target})


lifecycle = build_lifecycle(events)
same_activation = lifecycle.merge(subscribers, on="customer_id", suffixes=("_events", "_book"), validate="1:1")
require((same_activation.activation_date_events == same_activation.activation_date_book).all(),
        "activation date in events differs from subscribers_clean")

labels = pd.concat([churn_target(active_base(lifecycle, T), lifecycle, T) for T in ALL_DATES], ignore_index=True)
labels.groupby("snapshot_at").agg(n_active=("customer_id", "size"), n_churn=("churn_target", "sum"),
                                  n_unknown=("churn_target", lambda s: int(s.isna().sum())))

,n_active,n_churn,n_unknown
snapshot_at,,,
2025-12-01 00:00:00+03:00,1969,48,0
2026-01-01 00:00:00+03:00,1971,50,0
2026-02-01 00:00:00+03:00,1971,52,0
2026-03-01 00:00:00+03:00,1969,54,0
2026-04-01 00:00:00+03:00,1965,56,0
2026-05-01 00:00:00+03:00,1959,58,0
2026-06-01 00:00:00+03:00,1951,60,0
2026-07-01 00:00:00+03:00,1941,62,0
2026-08-01 00:00:00+03:00,1929,63,0


*Шаг 1. Активная база и метка*

---

## Шаг 2. Признаки договора и биллинга

Общее правило всех источников: **сначала** отбираются строки с `available_at ≤ T`, **потом** — месяцы окна. Обратный порядок незаметно пропустил бы строку прошлого месяца, опубликованную позже T. Каждый источник передаётся в функцию явно (`src`), поэтому в проверках шага 6 те же функции запускаются на искусственно изменённых копиях.

Окно — три календарных месяца `M−3, M−2, M−1` до T. Месяцы до подключения строк не имеют и **не заполняются нулями**: суммы и средние считаются по известным строкам, рядом стоит их число `n_billing_months_3m`. Абонент без строки не удаляется — его признаки остаются пропусками.

| Признак | Формула | Почему |
|---|---|---|
| `tenure_days` | `T − activation_date`, дни | H4: стаж <365 дней — выше доля оттока |
| `tariff_prev` | тариф строки M−1 | тариф меняется, историческое значение — из биллинга (ответ заказчика 5) |
| `debt_prev_rub` | `closing_debt_rub` за M−1 | H1: самая сильная связь (7,5% против 1,4%) |
| `debt_months_3m` | число месяцев окна с долгом > 0 | направление 1 EDA: долг в динамике |
| `debt_change_rub` | долг M−1 − долг M−2; пусто, если любого месяца нет | рост долга отличает новую неоплату от старого долга |
| `paid_to_charged_3m` | `Σ paid / Σ charged` по окну; пусто при `Σ charged = 0` | платёжная дисциплина; может быть > 1 при погашении старого долга |
| `payment_delay_prev_days` | задержка оплаты M−1 | корреляция с меткой 0,13; почти дублирует долг (0,79) |
| `traffic_gb_mean_3m` | среднее известного трафика окна | контроль объёма потребления |
| `traffic_per_day_3m` | `Σ traffic / Σ active_days` по месяцам с известным трафиком | нормировка неполного месяца подключения (устав, разд. 5) |
| `traffic_change_gb` | трафик M−1 − трафик M−2; пусто, если любой неизвестен | падение потребления как ранний сигнал |

In [8]:
def billing_features(base: pd.DataFrame, billing: pd.DataFrame, T: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Billing features of the window months known at T; returns (features, audit) indexed by customer_id."""
    at = snapshot_moment(T)
    months = window_months(T)
    known = billing[billing.available_at <= at]             # availability first ...
    rows = known[known.billing_month.isin(months)].copy()  # ... then the window months
    rows["has_debt"] = rows.closing_debt_rub > 0
    rows["traffic_days"] = rows.active_days.astype(float).where(rows.traffic_gb.notna())
    prev = rows[rows.billing_month == months[-1]].set_index("customer_id")
    prev2 = rows[rows.billing_month == months[-2]].set_index("customer_id")
    g = rows.groupby("customer_id")

    n_months = g.size()
    traffic_sum = g.traffic_gb.sum(min_count=1)
    traffic_known = g.traffic_gb.count()
    charged = g.charged_rub.sum()
    f = pd.DataFrame(index=pd.Index(base.customer_id, name="customer_id"))
    f["tariff_prev"] = prev.tariff
    f["debt_prev_rub"] = prev.closing_debt_rub
    f["debt_months_3m"] = g.has_debt.sum().astype("Int64")
    f["debt_change_rub"] = prev.closing_debt_rub - prev2.closing_debt_rub    # NaN unless both months exist
    f["paid_to_charged_3m"] = g.paid_rub.sum() / charged.where(charged > 0)
    f["payment_delay_prev_days"] = prev.payment_delay_days
    f["traffic_gb_mean_3m"] = traffic_sum / traffic_known.where(traffic_known > 0)
    f["traffic_per_day_3m"] = traffic_sum / g.traffic_days.sum(min_count=1)
    f["traffic_change_gb"] = prev.traffic_gb - prev2.traffic_gb            # NaN if either value is unknown
    f["n_billing_months_3m"] = n_months.reindex(f.index).fillna(0).astype("Int64")
    f["n_traffic_known_3m"] = traffic_known.reindex(f.index).fillna(0).astype("Int64")

    a = pd.DataFrame(index=f.index)
    a["billing_first_month_used"] = g.billing_month.min()
    a["billing_last_month_used"] = g.billing_month.max()
    a["billing_max_available_at"] = g.available_at.max()
    a["billing_prev_month_known"] = f.index.isin(prev.index)
    a["closing_debt_restored_3m"] = g.closing_debt_restored.sum().reindex(f.index).fillna(0).astype("Int64")
    a["voice_minutes_invalid_3m"] = g.voice_minutes_invalid.sum().reindex(f.index).fillna(0).astype("Int64")
    return f, a


def static_features(base: pd.DataFrame, subscribers: pd.DataFrame, T: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Permanent reference fields (client answer 5) and tenure at T."""
    s = base.merge(subscribers, on="customer_id", how="left", validate="1:1").set_index("customer_id")
    f = s[["region", "acquisition_channel", "customer_segment"]].copy()
    f["tenure_days"] = (T - s.activation_date).dt.days.astype("Int64")
    a = s[["activation_date_restored"]].copy()
    return f, a

*Шаг 2. Признаки договора и биллинга*

---

## Шаг 3. Признаки поддержки

Обращения пересчитываются **на T** по `support_clean` — эталонный вариант задания: `created_at` внутри окна `[1-е число M−3, T)` и `available_at ≤ T`. Поэтому в счёт входят и обращения прошлых месяцев, пришедшие с задержкой, но известные на T. Сумма трёх месячных снимков `customer_month` этапа 2 их бы не учла: каждый снимок зафиксирован на границе своего месяца. Сколько обращений добавляет пересчёт, видно в `feature_audit.csv` (`support_late_tickets_3m`).

| Признак | Формула | Почему |
|---|---|---|
| `support_count_3m` | число обращений окна, известных на T | H2 не поддержан на одном месяце; EDA предложил окно 2–3 месяца |
| `support_billing_count_3m` | то же, тема `billing` | направление EDA: спор о начислениях связан с долгом |

Ноль здесь — «нет известных корректно связанных обращений», а не доказательство их отсутствия (словарь этапа 2). Месяцы окна до 01.11.2025 обслуживала legacy-система (событие B001); в сентябре 2025 в ней ~20% меньше обращений, чем в среднем в новой. Это касается окон T = 01.12.2025 и 01.01.2026; число legacy-месяцев окна записано в аудит.

In [9]:
LEGACY_MONTHS = set(support.loc[support.source_system == "legacy", "created_at"]
                    .dt.tz_localize(None).dt.to_period("M").dt.to_timestamp())


def support_features(base: pd.DataFrame, support: pd.DataFrame, T: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Tickets created inside the window and already available at T, recounted at T."""
    at = snapshot_moment(T)
    months = window_months(T)
    start = snapshot_moment(months[0])
    known = support[support.available_at <= at]
    rows = known[(known.created_at >= start) & (known.created_at < at)].copy()
    created_month = rows.created_at.dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()
    rows["late"] = rows.available_at > (created_month + pd.DateOffset(months=1)).map(snapshot_moment)
    g = rows.groupby("customer_id")

    idx = pd.Index(base.customer_id, name="customer_id")
    f = pd.DataFrame(index=idx)
    f["support_count_3m"] = g.size().reindex(idx).fillna(0).astype("Int64")
    f["support_billing_count_3m"] = (rows[rows.topic == "billing"].groupby("customer_id").size()
                                     .reindex(idx).fillna(0).astype("Int64"))
    a = pd.DataFrame(index=idx)
    a["support_window_start"] = start
    a["support_max_available_at"] = g.available_at.max()
    a["support_late_tickets_3m"] = g.late.sum().reindex(idx).fillna(0).astype("Int64")
    a["support_legacy_months_3m"] = sum(m in LEGACY_MONTHS for m in months)
    return f, a

*Шаг 3. Признаки поддержки*

---

## Шаг 4. Признаки цифрового источника

Фильтр `available_at ≤ T` применяется ко всей строке, включая статус. Для месяца M−1 получаются четыре состояния, они записаны в `digital_status_prev`:

| Состояние | Значение `digital_status_prev` | Показатели M−1 |
|---|---|---|
| отчёт известен, `observed` | `observed` | известны, ноль — измеренный ноль |
| отчёт известен, `not_enabled` | `not_enabled` | неизвестны: сбор не подключён |
| отчёт известен, `export_gap` | `export_gap` | неизвестны: выгрузка неполна |
| отчёт ещё не опубликован на T | `not_available` | неизвестны вместе со статусом |

Более старый отчёт вместо M−1 **не подставляется**. Информация из опубликованных M−2 и M−3 входит только в явные трёхмесячные признаки, с числом наблюдаемых месяцев рядом. Все показатели нормируются по `observed_days` — активным дням договора, а не по длине месяца.

| Признак | Формула | Почему |
|---|---|---|
| `digital_sessions_per_day_prev` | `app_sessions / observed_days` за M−1, только `observed` и известный на T | вовлечённость в кабинет перед датой расчёта |
| `digital_sessions_per_day_3m` | `Σ sessions / Σ observed_days` по наблюдаемым известным месяцам окна | то же, устойчивее к задержке публикации |
| `digital_payment_attempts_3m` | `Σ payment_attempts` по тем же месяцам | счётчик для доли ошибок; оплата через кабинет |
| `digital_payment_failure_rate_3m` | `Σ failures / Σ attempts`; пусто при `Σ attempts = 0` | гипотеза: неуспешная оплата → долг (H1) → уход |
| `digital_selfservice_per_day_3m` | `Σ selfservice_actions / Σ observed_days` | самообслуживание; синтетический счётчик, не заявка на расторжение |

Индикаторы состояния: `digital_status_prev` и `digital_observed_months_3m` (сколько месяцев окна известны и наблюдаемы). Цифровая гипотеза предварительная: EDA по этому источнику не проводился.

In [10]:
DIGITAL_STATES = ["observed", "not_enabled", "export_gap", "not_available"]


def digital_features(base: pd.DataFrame, digital: pd.DataFrame, T: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Digital usage known at T; unknown status and unknown counts are never turned into zero activity."""
    at = snapshot_moment(T)
    months = window_months(T)
    known = digital[digital.available_at <= at]            # the whole row, status included
    rows = known[known.usage_month.isin(months)]
    obs = rows[rows.observation_status == "observed"]
    prev = rows[rows.usage_month == months[-1]].set_index("customer_id")
    prev_obs = prev[prev.observation_status == "observed"]
    g = obs.groupby("customer_id")
    days = g.observed_days.sum().astype(float)
    attempts = g.payment_attempts.sum().astype(float)

    idx = pd.Index(base.customer_id, name="customer_id")
    f = pd.DataFrame(index=idx)
    f["digital_status_prev"] = prev.observation_status.reindex(idx).fillna("not_available")
    f["digital_sessions_per_day_prev"] = prev_obs.app_sessions.astype(float) / prev_obs.observed_days.astype(float)
    f["digital_sessions_per_day_3m"] = g.app_sessions.sum().astype(float) / days
    f["digital_payment_attempts_3m"] = attempts.astype("Int64")
    f["digital_payment_failure_rate_3m"] = g.payment_failures.sum().astype(float) / attempts.where(attempts > 0)
    f["digital_selfservice_per_day_3m"] = g.selfservice_actions.sum().astype(float) / days
    f["digital_observed_months_3m"] = g.size().reindex(idx).fillna(0).astype("Int64")

    a = pd.DataFrame(index=idx)
    a["digital_prev_month"] = months[-1]
    a["digital_rows_known_3m"] = rows.groupby("customer_id").size().reindex(idx).fillna(0).astype("Int64")
    a["digital_max_available_at"] = rows.groupby("customer_id").available_at.max()
    return f, a

*Шаг 4. Признаки цифрового источника*

---

## Шаг 5. Сборка слоёв и паспорт признаков

`build_features(T, src)` собирает признаки и аудит на одну дату. Соединения идут по индексу активной базы (`left`), поэтому база не размножается и не сокращается; это проверяется на каждой T.

Четыре слоя хранятся отдельно, ключ везде `(customer_id, snapshot_at)`, строки упорядочены по `snapshot_at`, затем `customer_id`:

- `features.csv` — ключи и признаки с исходными пропусками;
- `labels.csv` — ключи, метка и исключённый конец горизонта;
- `split.csv` — ключи и часть разбиения;
- `feature_audit.csv` — использованные периоды, максимальные `available_at` реально использованных строк, полнота и флаги качества этапа 2.

Флаги качества (`closing_debt_restored`, `voice_minutes_invalid`, `activation_date_restored`) — диагностика финального аудита: их историческая доступность не доказана, поэтому они только в аудите.

In [11]:
SOURCES = {"lifecycle": lifecycle, "subscribers": subscribers, "billing": billing, "support": support,
           "digital": digital}

FEATURE_COLUMNS = [
    # contract and segment
    "tenure_days", "tariff_prev", "customer_segment", "acquisition_channel", "region",
    # billing
    "debt_prev_rub", "debt_months_3m", "debt_change_rub", "paid_to_charged_3m", "payment_delay_prev_days",
    # usage
    "traffic_gb_mean_3m", "traffic_per_day_3m", "traffic_change_gb",
    # support
    "support_count_3m", "support_billing_count_3m",
    # digital
    "digital_sessions_per_day_prev", "digital_sessions_per_day_3m", "digital_payment_attempts_3m",
    "digital_payment_failure_rate_3m", "digital_selfservice_per_day_3m",
    # completeness and observation state
    "n_billing_months_3m", "n_traffic_known_3m", "digital_status_prev", "digital_observed_months_3m",
]
AUDIT_COLUMNS = [
    "billing_first_month_used", "billing_last_month_used", "billing_max_available_at", "billing_prev_month_known",
    "support_window_start", "support_max_available_at", "support_late_tickets_3m", "support_legacy_months_3m",
    "digital_prev_month", "digital_rows_known_3m", "digital_max_available_at",
    "closing_debt_restored_3m", "voice_minutes_invalid_3m", "activation_date_restored",
]
KEYS = ["customer_id", "snapshot_at"]


def build_features(T: pd.Timestamp, src: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features and audit of the active base at T. Only rows with available_at <= T reach any feature."""
    base = active_base(src["lifecycle"], T)
    parts = [static_features(base, src["subscribers"], T), billing_features(base, src["billing"], T),
             support_features(base, src["support"], T), digital_features(base, src["digital"], T)]
    features = pd.concat([f for f, _ in parts], axis=1, join="outer")
    audit = pd.concat([a for _, a in parts], axis=1, join="outer")
    require(len(features) == len(base) and features.index.equals(pd.Index(base.customer_id)),
            f"T={T:%Y-%m-%d}: joins changed the active base")
    features = features.reset_index().assign(snapshot_at=snapshot_moment(T))
    audit = audit.reset_index().assign(snapshot_at=snapshot_moment(T))
    return features[KEYS + FEATURE_COLUMNS], audit[KEYS + AUDIT_COLUMNS]


def build_all(src: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    built = [build_features(T, src) for T in ALL_DATES]
    return (pd.concat([f for f, _ in built], ignore_index=True),
            pd.concat([a for _, a in built], ignore_index=True))


features, feature_audit = build_all(SOURCES)
split = features[KEYS].assign(split=features.snapshot_at.dt.tz_localize(None).map(SPLIT_OF))
require(features[KEYS].equals(labels[KEYS]), "features and labels have different keys or row order")
require(split.split.notna().all(), "a row outside the configured split")

print(f"features: {features.shape}, feature_audit: {feature_audit.shape}, labels: {labels.shape}")
split.groupby("split", sort=False).agg(rows=("customer_id", "size"), customers=("customer_id", "nunique"),
                                       dates=("snapshot_at", "nunique"))

features: (19541, 26), feature_audit: (19541, 16), labels: (19541, 4)


,rows,customers,dates
split,,,
train,15696,2319,8
validation,1929,1929,1
test,1916,1916,1


### Паспорт признаков

Паспорт заполняется по колонкам шаблона `feature_catalog_template.csv` и пишется в `docs/feature_catalog.csv`. Группа `полнота` — индикаторы полноты и состояния наблюдения; они нужны, но не входят в минимум 12 содержательных признаков. Ноутбук проверяет, что паспорт описывает ровно колонки `features.csv`.

In [12]:
CATALOG_COLUMNS = ["feature_name", "group", "meaning", "formula", "source", "window", "unit", "available_when",
                   "missing_policy", "zero_policy", "eda_reason", "leakage_risk", "include_in_X"]
BILL_AT = "строки биллинга с available_at ≤ T (месяц M публикуется 1-го числа M+1 в 00:00)"
DIG_AT = "строки цифрового отчёта с available_at ≤ T; задержка публикации 0–36 ч"
SUP_AT = "обращения с available_at ≤ T"
W3 = "M−3…M−1 (3 календарных месяца до T)"
CATALOG = [
    # name, group, meaning, formula, source, window, unit, available_when, missing, zero, eda, leakage
    ("tenure_days", "договор/сегмент", "Стаж абонента на T", "T − activation_date", "subscribers_clean",
     "на T", "дни", "постоянное поле справочника; activation < T по определению базы",
     "не бывает: дата подключения известна у всех", "не бывает: activation < T, минимум 1 день",
     "H4: стаж <90 — 3,6%, ≥365 — 2,7% (данных недостаточно, но направление устойчиво внутри групп долга)",
     "нет: дата подключения в прошлом", "yes"),
    ("tariff_prev", "договор/сегмент", "Тариф в предыдущем месяце", "tariff строки M−1", "billing_clean",
     "M−1", "категория", BILL_AT, "пусто, если строки M−1 нет; в преобразовании — категория пропуска",
     "—", "Basic 2,5%, Comfort 3,0%, Premium 3,1%; тариф меняется, поэтому из биллинга",
     "тариф месяца T (строка T) недоступен на T", "yes"),
    ("customer_segment", "договор/сегмент", "Сегмент абонента", "customer_segment", "subscribers_clean",
     "постоянное", "категория", "постоянное поле (ответ заказчика 5)", "не бывает", "—",
     "контроль состава базы", "нет: поле не обновляется", "yes"),
    ("acquisition_channel", "договор/сегмент", "Канал подключения", "acquisition_channel", "subscribers_clean",
     "постоянное", "категория", "постоянное поле (ответ заказчика 5)", "не бывает", "—",
     "контроль состава базы", "нет: поле не обновляется", "yes"),
    ("region", "договор/сегмент", "Регион обслуживания", "region", "subscribers_clean", "постоянное", "категория",
     "постоянное поле (ответ заказчика 5)", "не бывает", "—",
     "доля оттока 2,4–3,2% по регионам; сетевой отчёт региона не добавлен — он повторяет регион (EDA разд. 4)",
     "нет: поле не обновляется", "yes"),
    ("debt_prev_rub", "биллинг", "Долг на конец предыдущего месяца", "closing_debt_rub строки M−1", "billing_clean",
     "M−1", "руб.", BILL_AT, "пусто, если строки M−1 нет; не считается нулём",
     "0 — долга нет (76,7% строк EDA)", "H1: долг >0 — 7,5%, =0 — 1,4%; во всех 19 срезах",
     "долг строки месяца T — будущее; 40 значений восстановлены на этапе 2 (флаг в аудите)", "yes"),
    ("debt_months_3m", "биллинг", "Сколько месяцев окна с долгом", "Σ[closing_debt_rub > 0] по известным строкам окна",
     "billing_clean", W3, "месяцы", BILL_AT, "пусто, если в окне нет строк; число строк — n_billing_months_3m",
     "0 — ни в одном известном месяце долга не было", "H1 и направление 1 EDA: долг в динамике",
     "нет: только опубликованные строки окна", "yes"),
    ("debt_change_rub", "биллинг", "Изменение долга за последний месяц", "долг M−1 − долг M−2", "billing_clean",
     "M−2, M−1", "руб.", BILL_AT, "пусто, если нет строки M−1 или M−2; более старый месяц не подставляется",
     "0 — долг не изменился", "H1: рост долга отделяет новую неоплату от давно накопленной",
     "нет", "yes"),
    ("paid_to_charged_3m", "биллинг", "Доля начисленного, оплаченная за окно", "Σ paid_rub / Σ charged_rub по окну",
     "billing_clean", W3, "доля", BILL_AT, "пусто при Σ charged = 0 или без строк окна (на данных не встречается)",
     "0 — за окно не было оплат; >1 — погашение старого долга, не ошибка",
     "корреляция оплаты с меткой −0,11; направление 1 EDA", "нет", "yes"),
    ("payment_delay_prev_days", "биллинг", "Задержка оплаты в предыдущем месяце", "payment_delay_days строки M−1",
     "billing_clean", "M−1", "дни", BILL_AT, "пусто, если строки M−1 нет", "0 — оплата без задержки",
     "корреляция с меткой 0,13; с долгом 0,79 — почти дублирует долг", "нет", "yes"),
    ("traffic_gb_mean_3m", "история использования", "Средний трафик за окно",
     "Σ traffic_gb / число месяцев с известным трафиком", "billing_clean", W3, "ГБ/мес.", BILL_AT,
     "пусто, если трафик неизвестен во всех месяцах окна; число известных — n_traffic_known_3m",
     "нулей в данных нет", "у ушедших и оставшихся почти не различается — контроль объёма", "нет", "yes"),
    ("traffic_per_day_3m", "история использования", "Трафик на активный день",
     "Σ traffic_gb / Σ active_days по месяцам, где трафик известен", "billing_clean", W3, "ГБ/день", BILL_AT,
     "пусто, если трафик неизвестен во всех месяцах окна; дни месяцев с неизвестным трафиком не входят в знаменатель",
     "нулей в данных нет", "устав, разд. 5: неполный месяц подключения нормируется на active_days",
     "неполный месяц расторжения (месяц T) не используется", "yes"),
    ("traffic_change_gb", "история использования", "Изменение трафика за последний месяц",
     "traffic_gb M−1 − traffic_gb M−2", "billing_clean", "M−2, M−1", "ГБ", BILL_AT,
     "пусто, если любой из двух месяцев неизвестен или отсутствует", "0 — трафик не изменился",
     "гипотеза: падение потребления перед уходом; в EDA по одному месяцу различий нет", "нет", "yes"),
    ("support_count_3m", "поддержка", "Обращения за окно, известные на T",
     "число обращений с created_at ∈ [M−3, T) и available_at ≤ T", "support_clean", W3, "шт.", SUP_AT,
     "не бывает пропуска", "0 — нет известных корректно связанных обращений, не доказательство их отсутствия",
     "H2 не поддержан на одном месяце (2,7% / 3,0% / 3,0%); EDA предложил окно 2–3 месяца",
     "обращения, опубликованные после T, не входят; legacy-месяцы до 01.11.2025 в окнах 12.2025 и 01.2026", "yes"),
    ("support_billing_count_3m", "поддержка", "Обращения по оплате и начислениям за окно",
     "то же, topic = billing", "support_clean", W3, "шт.", SUP_AT, "не бывает пропуска",
     "0 — нет известных обращений по оплате", "направление EDA: тема billing связана с долгом", "то же", "yes"),
    ("digital_sessions_per_day_prev", "цифровой источник", "Сессии кабинета на активный день в M−1",
     "app_sessions / observed_days, только observed", "digital_usage_monthly", "M−1", "сессий/день", DIG_AT,
     "пусто при not_enabled, export_gap и неопубликованном отчёте; состояние — digital_status_prev",
     "0 — измеренная нулевая активность (только observed)", "предварительная гипотеза: вовлечённость перед уходом",
     "отчёт M−1, опубликованный после 00:00 T, неизвестен; не заменяется M−2", "yes"),
    ("digital_sessions_per_day_3m", "цифровой источник", "Сессии кабинета на активный день за окно",
     "Σ app_sessions / Σ observed_days по observed-месяцам окна, известным на T", "digital_usage_monthly", W3,
     "сессий/день", DIG_AT, "пусто, если нет ни одного известного observed-месяца; их число — digital_observed_months_3m",
     "0 — измеренный ноль во всех наблюдаемых месяцах", "предварительная гипотеза", "нет", "yes"),
    ("digital_payment_attempts_3m", "цифровой источник", "Попытки оплаты в кабинете за окно",
     "Σ payment_attempts по observed-месяцам окна", "digital_usage_monthly", W3, "попыток", DIG_AT,
     "пусто, если нет известных observed-месяцев", "0 — оплат через кабинет не было (измерено)",
     "знаменатель доли ошибок; оплата по всем каналам — в биллинге", "нет", "yes"),
    ("digital_payment_failure_rate_3m", "цифровой источник", "Доля неуспешных попыток оплаты",
     "Σ payment_failures / Σ payment_attempts по observed-месяцам окна", "digital_usage_monthly", W3, "доля", DIG_AT,
     "пусто при Σ attempts = 0 или без observed-месяцев: 0 из 0 — не измеренная нулевая доля",
     "0 — попытки были, все успешны", "гипотеза: неуспешная оплата → долг (H1) → уход", "нет", "yes"),
    ("digital_selfservice_per_day_3m", "цифровой источник", "Действия самообслуживания на активный день",
     "Σ selfservice_actions / Σ observed_days по observed-месяцам окна", "digital_usage_monthly", W3,
     "действий/день", DIG_AT, "пусто, если нет известных observed-месяцев", "0 — измеренный ноль",
     "предварительная гипотеза; синтетический счётчик, не заявка на расторжение", "нет", "yes"),
    ("n_billing_months_3m", "полнота", "Число известных строк биллинга в окне", "count строк окна", "billing_clean",
     W3, "месяцы", BILL_AT, "не бывает пропуска", "0 — строк нет (у активной базы не встречается)",
     "устав: короткая история у 95–99 абонентов на дату; месяцы до подключения — не нули", "нет", "yes"),
    ("n_traffic_known_3m", "полнота", "Число месяцев окна с известным трафиком", "count(traffic_gb) по окну",
     "billing_clean", W3, "месяцы", BILL_AT, "не бывает пропуска", "0 — трафик неизвестен во всех месяцах",
     "пропуск трафика — 0,9% строк EDA; знаменатель traffic_gb_mean_3m", "нет", "yes"),
    ("digital_status_prev", "полнота", "Состояние цифрового отчёта M−1 на T",
     "observation_status известной строки M−1, иначе not_available", "digital_usage_monthly", "M−1", "категория",
     DIG_AT, "не бывает пропуска: неизвестный отчёт — отдельная категория not_available", "—",
     "требование задания: различать ноль, not_enabled, export_gap и поздний отчёт",
     "статус из будущих строк не восстанавливается", "yes"),
    ("digital_observed_months_3m", "полнота", "Число известных observed-месяцев в окне", "count observed-строк окна",
     "digital_usage_monthly", W3, "месяцы", DIG_AT, "не бывает пропуска", "0 — нет ни одного измеренного месяца",
     "знаменатель трёхмесячных цифровых признаков", "нет", "yes"),
]
feature_catalog = pd.DataFrame(CATALOG, columns=CATALOG_COLUMNS)
template_columns = list(read_csv(CATALOG_TEMPLATE).columns)
require(template_columns == CATALOG_COLUMNS, f"feature_catalog_template.csv columns changed: {template_columns}")
require(feature_catalog.feature_name.tolist() == FEATURE_COLUMNS, "catalog and FEATURE_COLUMNS differ")
substantive = feature_catalog[feature_catalog.group != "полнота"]
require(len(substantive) >= 12 and substantive.group.nunique() >= 4, "need >= 12 substantive features in >= 4 groups")

write_csv(feature_catalog, DOCS / "feature_catalog.csv")
write_csv(features, OUTPUT / "features.csv")
write_csv(labels, OUTPUT / "labels.csv")
write_csv(split, OUTPUT / "split.csv")
write_csv(feature_audit, OUTPUT / "feature_audit.csv")
feature_catalog.groupby("group", sort=False).feature_name.agg(["size", list])

,size,list
group,,
договор/сегмент,5,"[tenure_days, tariff_prev, customer_segment, a..."
биллинг,5,"[debt_prev_rub, debt_months_3m, debt_change_ru..."
история использования,3,"[traffic_gb_mean_3m, traffic_per_day_3m, traff..."
поддержка,2,"[support_count_3m, support_billing_count_3m]"
цифровой источник,5,"[digital_sessions_per_day_prev, digital_sessio..."
полнота,4,"[n_billing_months_3m, n_traffic_known_3m, digi..."


### Полнота признаков по частям разбиения

Доля пропусков считается по каждой части отдельно; это описание входа, целевые исходы validation и test здесь не читаются.

In [13]:
completeness = (features.drop(columns=KEYS).isna().groupby(split.split).mean().T.round(4)
                [["train", "validation", "test"]].rename_axis("feature").reset_index())
status_share = (pd.crosstab(features.digital_status_prev, split.split, normalize="columns").round(4)
                [["train", "validation", "test"]].reindex(DIGITAL_STATES))
write_csv(completeness, OUTPUT / "feature_missing_share.csv")
display(completeness[completeness[["train", "validation", "test"]].max(axis=1) > 0])
status_share

split,feature,train,validation,test
7,debt_change_rub,0.0255,0.0259,0.0261
10,traffic_gb_mean_3m,0.0004,0.0000,0.0000
11,traffic_per_day_3m,0.0004,0.0000,0.0000
12,traffic_change_gb,0.0438,0.0467,0.0470
15,digital_sessions_per_day_prev,0.6504,0.6506,0.6373
16,digital_sessions_per_day_3m,0.2768,0.2722,0.2730
17,digital_payment_attempts_3m,0.2768,0.2722,0.2730
18,digital_payment_failure_rate_3m,0.3266,0.3292,0.3215
19,digital_selfservice_per_day_3m,0.2768,0.2722,0.2730


split,train,validation,test
digital_status_prev,,,
observed,0.3496,0.3494,0.3627
not_enabled,0.1278,0.1239,0.1253
export_gap,0.0195,0.0244,0.0240
not_available,0.5031,0.5023,0.4880


### Первичная связь с меткой на train

Описание, а не отбор признаков: ранговая корреляция Спирмена с меткой по известным значениям и доля оттока среди строк с известным и пропущенным значением. Строки одного абонента повторяются по датам, поэтому это ориентир для плана моделирования, а не тест значимости. Validation и test здесь не участвуют.

In [14]:
train_view = features[split.split == "train"].merge(labels[KEYS + ["churn_target"]], on=KEYS, validate="1:1")
y_train = train_view.churn_target.astype(float)
link = []
for col in [c for c in FEATURE_COLUMNS if c not in ("tariff_prev", "customer_segment", "acquisition_channel",
                                                    "region", "digital_status_prev")]:
    values = train_view[col].astype(float)
    known = values.notna()
    link.append({"feature": col, "group": feature_catalog.set_index("feature_name").group[col],
                 "n_known": int(known.sum()), "n_missing": int((~known).sum()),
                 "spearman_with_target": round(values[known].corr(y_train[known], method="spearman"), 4),
                 "churn_rate_known": round(y_train[known].mean(), 4),
                 "churn_rate_missing": round(y_train[~known].mean(), 4) if (~known).any() else np.nan})
link = pd.DataFrame(link).sort_values("spearman_with_target", key=abs, ascending=False, ignore_index=True)
status_link = (train_view.groupby("digital_status_prev").churn_target.agg(n_rows="size", n_churn="sum")
               .reindex(DIGITAL_STATES).reset_index())
status_link["churn_rate"] = (status_link.n_churn / status_link.n_rows).round(4)
write_csv(link, OUTPUT / "train_feature_target_link.csv")
write_csv(status_link, OUTPUT / "train_digital_status_churn.csv")
display(status_link)
link

,digital_status_prev,n_rows,n_churn,churn_rate
0,observed,5487,166,0.0303
1,not_enabled,2006,48,0.0239
2,export_gap,306,6,0.0196
3,not_available,7897,220,0.0279


,feature,group,n_known,n_missing,spearman_with_target,churn_rate_known,churn_rate_missing
0,debt_prev_rub,биллинг,15696,0,0.1707,0.0280,NaN
1,debt_change_rub,биллинг,15296,400,0.1324,0.0279,0.0350
2,payment_delay_prev_days,биллинг,15696,0,0.1320,0.0280,NaN
3,paid_to_charged_3m,биллинг,15696,0,-0.1217,0.0280,NaN
4,debt_months_3m,биллинг,15696,0,0.1078,0.0280,NaN
5,digital_payment_attempts_3m,цифровой источник,11351,4345,0.0206,0.0284,0.0272
6,traffic_change_gb,история использования,15009,687,-0.0195,0.0281,0.0262
7,digital_payment_failure_rate_3m,цифровой источник,10570,5126,0.0192,0.0289,0.0263
8,digital_sessions_per_day_prev,цифровой источник,5487,10209,0.0151,0.0303,0.0268
9,support_billing_count_3m,поддержка,15696,0,-0.0107,0.0280,NaN


*Шаг 5. Сборка слоёв и паспорт признаков*

---

## Шаг 6. Проверки времени и полноты

Проверки из раздела «Минимальные проверки» задания. Большинство — на искусственно изменённых копиях источников: те же функции `build_features` пересчитывают признаки, результат сравнивается с исходным. Все результаты — в `output/checks.csv`.

### База и ключи

Активная база пересчитывается вторым, независимым путём: по `activation_date` справочника и расторжениям **сырого** журнала этапа 1, на множествах Python. Числа сверяются с таблицей снимков устава.

In [15]:
dups = int(features.duplicated(KEYS).sum())
record_check("key (customer_id, snapshot_at) is unique in features, labels, split, audit",
             dups == 0 and all(t[KEYS].equals(features[KEYS]) for t in (labels, split, feature_audit)),
             f"duplicates: {dups}; all four layers share keys and row order")

activation = {r.customer_id: r.activation_date.date() for r in subscribers.itertuples()}
raw_term = raw_events[raw_events.event_type == "termination"]
require(raw_term.customer_id.is_unique, "raw log: several terminations for one customer")
termination = {r.customer_id: r.event_date.date() for r in raw_term.itertuples()}
CHARTER = {"2025-12-01": (1969, 48), "2026-01-01": (1971, 50), "2026-02-01": (1971, 52), "2026-03-01": (1969, 54),
           "2026-04-01": (1965, 56), "2026-05-01": (1959, 58), "2026-06-01": (1951, 60), "2026-07-01": (1941, 62),
           "2026-08-01": (1929, 63), "2026-09-01": (1916, 65)}
independent = []
for T in ALL_DATES:
    t, end = T.date(), (T + pd.DateOffset(months=1)).date()
    expected = {c for c, a in activation.items() if a < t and (c not in termination or termination[c] >= t)}
    expected_churn = {c for c in expected if c in termination and t <= termination[c] < end}
    rows = features[features.snapshot_at == snapshot_moment(T)]
    lab = labels[labels.snapshot_at == snapshot_moment(T)]
    independent.append({"snapshot_at": snapshot_moment(T), "n_features": len(rows), "n_independent": len(expected),
                        "n_charter": CHARTER[f"{T:%Y-%m-%d}"][0], "same_customers": set(rows.customer_id) == expected,
                        "churn_labels": int(lab.churn_target.sum()), "churn_independent": len(expected_churn),
                        "churn_charter": CHARTER[f"{T:%Y-%m-%d}"][1],
                        "same_churners": set(lab.customer_id[lab.churn_target == 1]) == expected_churn})
independent = pd.DataFrame(independent)
write_csv(independent, OUTPUT / "check_active_base.csv")
base_ok = (independent.same_customers.all() and independent.same_churners.all()
           and (independent.n_features == independent.n_charter).all()
           and (independent.churn_labels == independent.churn_charter).all())
record_check("rows per T equal an independent recount from the raw log and the charter table", bool(base_ok),
             f"{len(independent)} dates; {independent.n_features.sum()} rows, {independent.churn_labels.sum()} churn")

last_end = {part: labels.target_end_exclusive[split.split == part].max() for part in ("train", "validation")}
record_check("labels of each part are fully observed before the next part starts",
             bool(last_end["train"] <= snapshot_moment(VALIDATION_DATES[0])
                  and last_end["validation"] <= snapshot_moment(TEST_DATES[0])
                  and labels.churn_target.notna().all()),
             f"train ends {last_end['train']:%Y-%m-%d}, validation ends {last_end['validation']:%Y-%m-%d}")

used = feature_audit
availability_ok = all((used[c].dropna() <= used.snapshot_at[used[c].notna()]).all()
                      for c in ("billing_max_available_at", "support_max_available_at", "digital_max_available_at"))
record_check("every used row of every source has available_at <= snapshot_at", bool(availability_ok),
             "max available_at of billing, support and digital rows in feature_audit.csv")
leaking = [c for c in features.columns if "termination" in c or c in {"churn_target", "target_end_exclusive"}]
record_check("no label or termination column in features.csv", not leaking, f"columns: {leaking or 'none'}")
independent

,snapshot_at,n_features,n_independent,n_charter,same_customers,churn_labels,churn_independent,churn_charter,same_churners
0,2025-12-01 00:00:00+03:00,1969,1969,1969,True,48,48,48,True
1,2026-01-01 00:00:00+03:00,1971,1971,1971,True,50,50,50,True
2,2026-02-01 00:00:00+03:00,1971,1971,1971,True,52,52,52,True
3,2026-03-01 00:00:00+03:00,1969,1969,1969,True,54,54,54,True
4,2026-04-01 00:00:00+03:00,1965,1965,1965,True,56,56,56,True
5,2026-05-01 00:00:00+03:00,1959,1959,1959,True,58,58,58,True
6,2026-06-01 00:00:00+03:00,1951,1951,1951,True,60,60,60,True
7,2026-07-01 00:00:00+03:00,1941,1941,1941,True,62,62,62,True
8,2026-08-01 00:00:00+03:00,1929,1929,1929,True,63,63,63,True
9,2026-09-01 00:00:00+03:00,1916,1916,1916,True,65,65,65,True


### Будущие записи и будущая метка не меняют признаки

На каждую из 10 дат T в копиях источников меняется **всё, что опубликовано после T**, и добавляются новые будущие записи:

- биллинг: у строк с `available_at > T` долг +10 000 руб., трафик ×10, тариф `Ultra`; добавлены строки за декабрь 2026;
- поддержка: у обращений с `available_at > T` тема меняется на `billing`; добавлены 200 обращений, **созданных в M−1** (внутри окна), но опубликованных через час после T;
- цифровой источник: строки с `available_at > T` становятся `observed` с 99 сессиями и 9 ошибками оплаты.

Отдельно меняются будущие метки: у ушедших в горизонте расторжение переносится на полгода, у 30 оставшихся появляется расторжение внутри горизонта. Абоненты остаются в базе (`termination ≥ T`), метки меняются, признаки — нет.

In [16]:
def with_future_changes(src: dict, T: pd.Timestamp) -> dict:
    """Copy of the sources where everything published after T is altered and new future records are added."""
    at = snapshot_moment(T)
    out = dict(src)

    b = src["billing"].copy()
    late = b.available_at > at
    b.loc[late, "closing_debt_rub"] += 10_000
    b.loc[late, "traffic_gb"] *= 10
    b.loc[late, "tariff"] = "Ultra"
    extra = b.drop_duplicates("customer_id").head(100).assign(billing_month=pd.Timestamp("2026-12-01"),
                                                             available_at=snapshot_moment(pd.Timestamp("2027-01-01")))
    out["billing"] = pd.concat([b, extra], ignore_index=True)

    s = src["support"].copy()
    s.loc[s.available_at > at, "topic"] = "billing"
    base_ids = active_base(src["lifecycle"], T).customer_id.head(200)
    m1_start = snapshot_moment(window_months(T)[-1])
    new = pd.DataFrame({"ticket_id": [f"FUT{i:05d}" for i in range(len(base_ids))], "customer_id": base_ids.values,
                        "source_system": "current", "created_at": m1_start + pd.Timedelta(3, unit="D"),
                        "available_at": at + pd.Timedelta(1, unit="h"), "topic": "billing"})
    out["support"] = pd.concat([s, new], ignore_index=True)

    d = src["digital"].copy()
    late = d.available_at > at
    d.loc[late, ["observation_status"]] = "observed"
    d.loc[late, "observed_days"] = d.loc[late, "observed_days"].fillna(28)
    d.loc[late, ["app_sessions", "payment_attempts", "payment_failures"]] = [99, 9, 9]
    out["digital"] = d
    return out


def with_future_labels(src: dict, T: pd.Timestamp) -> dict:
    """Copy where the outcomes inside the horizon of T are changed while the base at T stays the same."""
    lc = src["lifecycle"].copy()
    end = T + pd.DateOffset(months=1)
    churn = (lc.termination_date >= T) & (lc.termination_date < end)
    lc.loc[churn, "termination_date"] = lc.loc[churn, "termination_date"] + pd.DateOffset(months=6)
    stayers = lc[(lc.activation_date < T) & lc.termination_date.isna()].index[:30]
    lc.loc[stayers, "termination_date"] = T + pd.Timedelta(10, unit="D")
    return dict(src, lifecycle=lc)


future_rows, label_rows = [], []
for T in ALL_DATES:
    reference, reference_audit = build_features(T, SOURCES)
    changed, _ = build_features(T, with_future_changes(SOURCES, T))
    future_rows.append({"snapshot_at": snapshot_moment(T), "same": changed.equals(reference)})
    relabelled_src = with_future_labels(SOURCES, T)
    relabelled, _ = build_features(T, relabelled_src)
    base_T = active_base(lifecycle, T)
    labels_changed = not churn_target(base_T, relabelled_src["lifecycle"], T).equals(churn_target(base_T, lifecycle, T))
    label_rows.append({"snapshot_at": snapshot_moment(T), "same": relabelled.equals(reference),
                       "labels_changed": labels_changed})

future_rows, label_rows = pd.DataFrame(future_rows), pd.DataFrame(label_rows)
record_check("features at T do not change when records published after T are altered or added",
             bool(future_rows.same.all()), f"{len(future_rows)} dates; billing, support (incl. late tickets of M-1), digital")
record_check("changing future labels does not change features", bool(label_rows.same.all() and label_rows.labels_changed.all()),
             f"{len(label_rows)} dates; labels differ, features equal")

### Граница доступности: ровно T включается, на секунду позже — нет

Дата T = 01.07.2026. Для каждого источника берётся реальная строка, её `available_at` ставится ровно на T и на T + 1 секунду:

- биллинг: строка июня абонента A (по данным уже ровно 00:00 01.07);
- поддержка: новое обращение абонента B, созданное 15 июня;
- цифровой источник: июньский отчёт абонента C (`observed`, опубликован с задержкой) и абонента D (опубликован ровно в T).

In [17]:
T = pd.Timestamp("2026-07-01")
at = snapshot_moment(T)
reference, _ = build_features(T, SOURCES)
ref = reference.set_index("customer_id")
m1 = window_months(T)[-1]


def shifted(table_name: str, mask: pd.Series, column: str, moment: pd.Timestamp) -> dict:
    t = SOURCES[table_name].copy()
    t.loc[mask, column] = moment
    return dict(SOURCES, **{table_name: t})


def row_at(src: dict, customer: str) -> pd.Series:
    return build_features(T, src)[0].set_index("customer_id").loc[customer]


# Billing: the June row of A is published exactly at T in the data
A = ref.index[ref.n_billing_months_3m == 3][0]
mask_A = (billing.customer_id == A) & (billing.billing_month == m1)
require(billing.loc[mask_A, "available_at"].eq(at).all(), "row of A is expected at exactly T")
on_A, after_A = row_at(shifted("billing", mask_A, "available_at", at), A), \
                row_at(shifted("billing", mask_A, "available_at", at + pd.Timedelta(1, unit="s")), A)
billing_ok = (on_A.debt_prev_rub == ref.loc[A, "debt_prev_rub"] and on_A.n_billing_months_3m == 3
              and pd.isna(after_A.debt_prev_rub) and pd.isna(after_A.tariff_prev) and after_A.n_billing_months_3m == 2)

# Support: a new ticket of B created inside M-1
B = ref.index[1]
ticket = pd.DataFrame({"ticket_id": ["EDGE001"], "customer_id": [B], "source_system": ["current"],
                       "created_at": [snapshot_moment(m1) + pd.Timedelta(14, unit="D")], "available_at": [at],
                       "topic": ["billing"]})
src_on = dict(SOURCES, support=pd.concat([support, ticket], ignore_index=True))
src_after = dict(SOURCES, support=pd.concat([support, ticket.assign(available_at=at + pd.Timedelta(1, unit="s"))],
                                            ignore_index=True))
support_ok = (row_at(src_on, B).support_count_3m == ref.loc[B, "support_count_3m"] + 1
              and row_at(src_after, B).support_count_3m == ref.loc[B, "support_count_3m"])

# Digital: C is observed and late (unknown at T); D is published exactly at T
june = digital[(digital.usage_month == m1) & digital.customer_id.isin(ref.index)]
C = june[(june.observation_status == "observed") & (june.available_at > at)].customer_id.iloc[0]
D = june[(june.observation_status == "observed") & (june.available_at == at)].customer_id.iloc[0]
mask_C = (digital.customer_id == C) & (digital.usage_month == m1)
mask_D = (digital.customer_id == D) & (digital.usage_month == m1)
on_C = row_at(shifted("digital", mask_C, "available_at", at), C)
after_D = row_at(shifted("digital", mask_D, "available_at", at + pd.Timedelta(1, unit="s")), D)
digital_ok = (ref.loc[C, "digital_status_prev"] == "not_available" and pd.isna(ref.loc[C, "digital_sessions_per_day_prev"])
              and on_C.digital_status_prev == "observed" and pd.notna(on_C.digital_sessions_per_day_prev)
              and ref.loc[D, "digital_status_prev"] == "observed"
              and after_D.digital_status_prev == "not_available" and pd.isna(after_D.digital_sessions_per_day_prev))

record_check("availability boundary: available_at = T is included, T + 1 s is excluded",
             bool(billing_ok and support_ok and digital_ok),
             f"billing {A}: {billing_ok}; support {B}: {support_ok}; digital {C}/{D}: {digital_ok}")

### Короткая история и пропущенные месяцы

- Число строк окна должно равняться числу месяцев окна с момента подключения (и с начала выгрузки, сентябрь 2025). Тогда история из одного месяца остаётся одним месяцем, а не тремя с нулями.
- При одном месяце истории разности неизвестны, средний трафик равен трафику единственного месяца.
- Реальные строки, где трафик M−2 неизвестен: разность трафика пустая, хотя предыдущие месяцы известны.
- Искусственно удалённая строка M−2: разность долга пустая, а не «M−1 минус M−3».

In [18]:
first_billing_month = billing.billing_month.min()
f = features.merge(subscribers[["customer_id", "activation_date"]], on="customer_id", validate="m:1")
T_naive = f.snapshot_at.dt.tz_localize(None)
start = f.activation_date.dt.to_period("M").dt.to_timestamp().clip(lower=first_billing_month)
expected_months = np.clip((T_naive.dt.year - start.dt.year) * 12 + (T_naive.dt.month - start.dt.month), 0, WINDOW)
short = f[f.n_billing_months_3m < WINDOW]
record_check("window rows = months since activation; short history is not padded",
             bool((f.n_billing_months_3m == expected_months).all()),
             f"{len(short)} rows with < {WINDOW} months: "
             f"{ {int(k): int(v) for k, v in short.n_billing_months_3m.value_counts().sort_index().items()} }")

one = features[features.n_billing_months_3m == 1]
one_prev = one.merge(billing.assign(snapshot_at=(billing.billing_month + pd.DateOffset(months=1)).map(snapshot_moment)),
                     on=["customer_id", "snapshot_at"], how="left", validate="1:1")
one_ok = (one.debt_change_rub.isna().all() and one.traffic_change_gb.isna().all()
          and np.allclose(one_prev.traffic_gb_mean_3m.dropna(), one_prev.traffic_gb.dropna())
          and one.debt_months_3m.le(1).all())
record_check("one-month history: changes unknown, 3-month mean equals that month", bool(one_ok),
             f"{len(one)} rows with one month of history")

prev_traffic = billing.assign(snapshot_at=(billing.billing_month + pd.DateOffset(months=2)).map(snapshot_moment))
gap = features.merge(prev_traffic[["customer_id", "snapshot_at", "traffic_gb"]], on=KEYS, how="left", validate="1:1")
gap = gap[gap.traffic_gb.isna() & (gap.n_billing_months_3m >= 2)]
record_check("unknown traffic of M-2 gives an unknown change, not a substitute month",
             bool(gap.traffic_change_gb.isna().all() and len(gap) > 0), f"{len(gap)} real rows with unknown M-2 traffic")

E = ref.index[ref.n_billing_months_3m == 3][1]
no_m2 = dict(SOURCES, billing=billing[~((billing.customer_id == E) & (billing.billing_month == window_months(T)[-2]))])
row_E = row_at(no_m2, E)
record_check("a removed month M-2 is not replaced by M-3 in the debt change",
             bool(pd.isna(row_E.debt_change_rub) and row_E.n_billing_months_3m == 2 and pd.notna(row_E.debt_prev_rub)),
             f"{E} at {T:%Y-%m-%d}: debt_change empty, 2 months in the window")

### Ноль и неизвестное в цифровом источнике

- `not_enabled`, `export_gap` и неопубликованный отчёт не дают нулевых сессий: показатель M−1 пустой.
- У `observed` нулевые сессии сохраняются как измеренный ноль.
- Ноль попыток оплаты за окно даёт пустую долю ошибок при известном счётчике попыток 0.

In [19]:
unobserved = features[features.digital_status_prev != "observed"]
zero_sessions = features[(features.digital_status_prev == "observed") & (features.digital_sessions_per_day_prev == 0)]
zero_attempts = features[features.digital_payment_attempts_3m == 0]
no_observed = features[features.digital_observed_months_3m == 0]
digital_zero_ok = (unobserved.digital_sessions_per_day_prev.isna().all()
                   and features.loc[features.digital_status_prev == "observed", "digital_sessions_per_day_prev"].notna().all()
                   and len(zero_sessions) > 0
                   and zero_attempts.digital_payment_failure_rate_3m.isna().all() and len(zero_attempts) > 0
                   and no_observed[["digital_sessions_per_day_3m", "digital_payment_attempts_3m",
                                    "digital_payment_failure_rate_3m"]].isna().all().all())
record_check("not_enabled, export_gap and late reports are unknown, not zero; 0 attempts gives an unknown rate",
             bool(digital_zero_ok),
             f"unknown M-1: {unobserved.digital_status_prev.value_counts().to_dict()}; measured zero sessions: "
             f"{len(zero_sessions)}; zero attempts with unknown rate: {len(zero_attempts)}")
pd.DataFrame(CHECKS).tail(8)

,check,passed,detail
14,features at T do not change when records publi...,True,"10 dates; billing, support (incl. late tickets..."
15,changing future labels does not change features,True,"10 dates; labels differ, features equal"
16,availability boundary: available_at = T is inc...,True,billing C000001: True; support C000002: True; ...
17,window rows = months since activation; short h...,True,"980 rows with < 3 months: {1: 500, 2: 480}"
18,"one-month history: changes unknown, 3-month me...",True,500 rows with one month of history
19,unknown traffic of M-2 gives an unknown change...,True,187 real rows with unknown M-2 traffic
20,a removed month M-2 is not replaced by M-3 in ...,True,"C000002 at 2026-07-01: debt_change empty, 2 mo..."
21,"not_enabled, export_gap and late reports are u...",True,"unknown M-1: {'not_available': 9801, 'not_enab..."


*Шаг 6. Проверки времени и полноты*

---

## Шаг 7. Преобразование данных для кандидата 1

Первый кандидат — логистическая регрессия (обоснование — в `docs/modeling_plan.md`). Ей нужны числовая матрица без пропусков и сопоставимые масштабы. Преобразователь — `make_preprocessor` из `starter/preprocessing_template.py`, со **своими** явными списками столбцов:

- числа: медиана train + индикатор пропуска (для признаков, у которых на train были пропуски), затем стандартизация;
- категории: пропуск → категория `__MISSING__`, One-Hot Encoding; новая категория на отложенных данных даёт нулевой вектор (`handle_unknown='ignore'`).

Медианы, масштабы и категории вычисляются **только на train**. Validation и test проходят только `transform`; их метки здесь не читаются. Матрицы в CSV не сохраняются: строки `X_part` идут в том же порядке, что строки `features.csv` этой части (`snapshot_at`, затем `customer_id`).

Если на следующем этапе появятся временные фолды внутри train, преобразователь будет обучаться заново внутри каждого фолда.

In [20]:
spec = importlib.util.spec_from_file_location("preprocessing_template", PREPROCESSING_TEMPLATE)
preprocessing_template = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preprocessing_template)

CATEGORICAL_X = ["tariff_prev", "customer_segment", "acquisition_channel", "region", "digital_status_prev"]
NUMERIC_X = [c for c in feature_catalog.feature_name[feature_catalog.include_in_X == "yes"] if c not in CATEGORICAL_X]
require(set(NUMERIC_X) | set(CATEGORICAL_X) <= set(FEATURE_COLUMNS), "X lists reference unknown columns")
require(not (set(NUMERIC_X) | set(CATEGORICAL_X)) & {"customer_id", "snapshot_at", "split"}, "keys in X lists")


def model_input(part: pd.DataFrame) -> pd.DataFrame:
    """Explicit X columns only; nullable integers become float so missing values are NaN."""
    return pd.concat([part[NUMERIC_X].astype(float), part[CATEGORICAL_X].astype(object)], axis=1)


parts = {name: features[split.split == name].reset_index(drop=True) for name in ("train", "validation", "test")}
inputs = {name: model_input(part) for name, part in parts.items()}

prep = preprocessing_template.make_preprocessor(NUMERIC_X, CATEGORICAL_X)
X = {"train": prep.fit_transform(inputs["train"])}
fitted_state = json.dumps({"medians": prep.named_transformers_["numeric"].named_steps["impute"].statistics_.tolist(),
                           "scale": prep.named_transformers_["numeric"].named_steps["scale"].scale_.tolist(),
                           "categories": [c.tolist() for c in
                                          prep.named_transformers_["categorical"].named_steps["encode"].categories_]})
X["validation"] = prep.transform(inputs["validation"])
X["test"] = prep.transform(inputs["test"])

x_columns = pd.DataFrame({"x_column": prep.get_feature_names_out()})
x_columns["source_feature"] = x_columns.x_column.str.split("__", n=1).str[1]
x_columns["kind"] = np.select([x_columns.x_column.str.startswith("numeric__missingindicator_"),
                               x_columns.x_column.str.startswith("numeric__")],
                              ["флаг пропуска (train)", "число, медиана train + стандартизация"], "one-hot категории")
x_columns["source_feature"] = np.where(x_columns.kind == "флаг пропуска (train)",
                                       x_columns.x_column.str.removeprefix("numeric__missingindicator_"),
                                       x_columns.source_feature)
x_columns.loc[x_columns.kind == "one-hot категории", "source_feature"] = [
    next(c for c in CATEGORICAL_X if name.startswith(f"categorical__{c}_"))
    for name in x_columns.x_column[x_columns.kind == "one-hot категории"]]

shapes = pd.DataFrame([{"part": name, "dates": ", ".join(f"{d:%Y-%m-%d}" for d in
                                                       parts[name].snapshot_at.drop_duplicates()),
                        "rows": X[name].shape[0], "x_columns": X[name].shape[1],
                        "customers": parts[name].customer_id.nunique(),
                        "row_order": "как в features.csv: snapshot_at, затем customer_id",
                        "first_key": f"{parts[name].customer_id.iloc[0]} @ {parts[name].snapshot_at.iloc[0]:%Y-%m-%d}",
                        "last_key": f"{parts[name].customer_id.iloc[-1]} @ {parts[name].snapshot_at.iloc[-1]:%Y-%m-%d}",
                        "all_finite": bool(np.isfinite(X[name]).all())} for name in X])
write_csv(x_columns, OUTPUT / "x_columns.csv")
write_csv(shapes, OUTPUT / "matrix_shapes.csv")
print(f"X columns: {len(x_columns)} from {len(NUMERIC_X)} numeric and {len(CATEGORICAL_X)} categorical features")
display(x_columns.kind.value_counts())
shapes

X columns: 45 from 19 numeric and 5 categorical features


kind
число, медиана train + стандартизация    19
one-hot категории                        17
флаг пропуска (train)                     9
Name: count, dtype: int64

,part,dates,rows,x_columns,customers,row_order,first_key,last_key,all_finite
0,train,"2025-12-01, 2026-01-01, 2026-02-01, 2026-03-01...",15696,45,2319,"как в features.csv: snapshot_at, затем custome...",C000001 @ 2025-12-01,C002450 @ 2026-07-01,True
1,validation,2026-08-01,1929,45,1929,"как в features.csv: snapshot_at, затем custome...",C000001 @ 2026-08-01,C002500 @ 2026-08-01,True
2,test,2026-09-01,1916,45,1916,"как в features.csv: snapshot_at, затем custome...",C000001 @ 2026-09-01,C002550 @ 2026-09-01,True


### Проверки преобразования

- Статистики вычислены на train: медианы преобразователя равны медианам train, посчитанным напрямую pandas; после `transform` validation и test состояние преобразователя не изменилось.
- Новые категории и пропуск категории в validation обрабатываются без ошибки и без `fit`; строки с ними получают нулевой one-hot вектор по этому признаку, остальные столбцы не меняются.
- Порядок строк: перевёрнутая validation после `transform` и обратного переворота даёт ту же матрицу — строка `X` зависит только от своей строки признаков.

In [21]:
numeric_impute = prep.named_transformers_["numeric"].named_steps["impute"]
train_medians = inputs["train"][NUMERIC_X].median()
state_after = json.dumps({"medians": numeric_impute.statistics_.tolist(),
                          "scale": prep.named_transformers_["numeric"].named_steps["scale"].scale_.tolist(),
                          "categories": [c.tolist() for c in
                                         prep.named_transformers_["categorical"].named_steps["encode"].categories_]})
record_check("preprocessor statistics come from train only and do not change on transform",
             bool(np.allclose(numeric_impute.statistics_, train_medians.to_numpy()) and state_after == fitted_state),
             f"{len(NUMERIC_X)} medians equal train medians; scale and categories unchanged after transform")

novel = inputs["validation"].copy()
novel.loc[0, "tariff_prev"] = "Ultra"
novel.loc[1, "region"] = "Новая область"
novel.loc[2, "tariff_prev"] = np.nan
novel.loc[3, "digital_status_prev"] = "pending_migration"
X_novel = prep.transform(novel)
tariff_cols = x_columns.index[x_columns.source_feature == "tariff_prev"]
region_cols = x_columns.index[x_columns.source_feature == "region"]
other_cols = x_columns.index[~x_columns.source_feature.isin(["tariff_prev", "region", "digital_status_prev"])]
novel_ok = (X_novel.shape == X["validation"].shape and np.isfinite(X_novel).all()
            and X_novel[0, tariff_cols].sum() == 0 and X_novel[1, region_cols].sum() == 0
            and np.allclose(X_novel[:4][:, other_cols], X["validation"][:4][:, other_cols])
            and np.allclose(X_novel[4:], X["validation"][4:]) and state_after == fitted_state)
record_check("unseen categories and a missing category in validation transform without refit", bool(novel_ok),
             "Ultra / Новая область / missing tariff / unseen digital status -> zero one-hot, other rows unchanged")

reversed_X = prep.transform(inputs["validation"].iloc[::-1])[::-1]
record_check("transform keeps rows aligned with keys",
             bool(np.allclose(reversed_X, X["validation"]) and len(X["validation"]) == len(parts["validation"])),
             f"validation {X['validation'].shape}, row i = features row i of the validation part")

*Шаг 7. Преобразование данных для кандидата 1*

---

## Шаг 8. Базовое правило на train

Правило: на каждой T активные абоненты сортируются по убыванию `debt_prev_rub` (долг на конец M−1, строка доступна на T); при равном долге — `customer_id` по возрастанию. Неизвестный долг ставится **после** всех известных и нулём не считается (в уставе было «пустой долг = 0»; на данных неизвестного долга нет, поэтому числа не меняются). В список входят первые `K = floor(0,10 × N_active)`, где N — вся активная база на T, а не только строки с полными признаками.

- `Recall@K = TP@K / число ушедших на T`, `Precision@K = TP@K / K`. Если ушедших нет или K = 0, соответствующая доля неизвестна (пусто).
- Усреднение по датам двумя способами: **pooled** — сумма числителей / сумма знаменателей (каждый уход весит одинаково); **mean of dates** — среднее долей по датам с определённой долей (каждая дата весит одинаково).

Считается только на train. Метки validation и test не читаются.

In [22]:
def debt_ranking(day: pd.DataFrame) -> pd.DataFrame:
    """Debt of M-1 descending, ties by customer_id ascending, unknown debt after every known value."""
    return (day.assign(_debt_unknown=day.debt_prev_rub.isna())
            .sort_values(["_debt_unknown", "debt_prev_rub", "customer_id"], ascending=[True, False, True],
                         kind="mergesort")
            .drop(columns="_debt_unknown").reset_index(drop=True))


def campaign_size(n_active: int) -> int:
    return math.floor(CAMPAIGN_FRACTION * n_active)


train_rows = (features.loc[split.split == "train", KEYS + ["debt_prev_rub"]]
              .merge(labels.loc[split.split == "train", KEYS + ["churn_target"]], on=KEYS, how="inner", validate="1:1"))
metrics = []
for snapshot_at, day in train_rows.groupby("snapshot_at"):
    ranked = debt_ranking(day)
    n, k = len(ranked), campaign_size(len(ranked))
    positives = int(ranked.churn_target.sum())
    hits = int(ranked.churn_target.iloc[:k].sum())
    cutoff = ranked.debt_prev_rub.iloc[k - 1] if k else np.nan
    metrics.append({
        "snapshot_at": f"{snapshot_at:%Y-%m-%d}", "n_active": n, "k": k, "n_unknown_debt": int(ranked.debt_prev_rub.isna().sum()),
        "n_debt_positive": int((ranked.debt_prev_rub > 0).sum()), "debt_at_cutoff_rub": cutoff,
        "n_tied_at_cutoff": int((ranked.debt_prev_rub == cutoff).sum()),
        "n_tied_inside_k": int((ranked.debt_prev_rub.iloc[:k] == cutoff).sum()),
        "tp_at_k": hits, "n_positive": positives,
        "recall_at_k": hits / positives if positives else np.nan, "precision_at_k": hits / k if k else np.nan,
    })
baseline = pd.DataFrame(metrics)
COUNT_COLUMNS = ["n_active", "k", "n_unknown_debt", "n_debt_positive", "n_tied_at_cutoff", "n_tied_inside_k",
                 "tp_at_k", "n_positive"]

defined_recall, defined_precision = baseline.n_positive > 0, baseline.k > 0
summary = pd.DataFrame([
    {"snapshot_at": "pooled", "n_active": baseline.n_active.sum(), "k": baseline.k.sum(),
     "tp_at_k": baseline.tp_at_k.sum(), "n_positive": baseline.n_positive.sum(),
     "recall_at_k": baseline.tp_at_k.sum() / baseline.n_positive[defined_recall].sum(),
     "precision_at_k": baseline.tp_at_k.sum() / baseline.k[defined_precision].sum()},
    {"snapshot_at": "mean_of_dates", "recall_at_k": baseline.recall_at_k[defined_recall].mean(),
     "precision_at_k": baseline.precision_at_k[defined_precision].mean()},
])
baseline_table = pd.concat([baseline.assign(row_type="date"), summary.assign(row_type="average")], ignore_index=True)
baseline_table[COUNT_COLUMNS] = baseline_table[COUNT_COLUMNS].astype("Int64")
baseline_table[["recall_at_k", "precision_at_k"]] = baseline_table[["recall_at_k", "precision_at_k"]].round(6)
write_csv(baseline_table, OUTPUT / "baseline_train_metrics.csv")
baseline_table

,snapshot_at,n_active,k,n_unknown_debt,n_debt_positive,debt_at_cutoff_rub,n_tied_at_cutoff,n_tied_inside_k,tp_at_k,n_positive,recall_at_k,precision_at_k,row_type
0,2025-12-01,1969,196,0,455,624.47,1,1,16,48,0.333333,0.081633,date
1,2026-01-01,1971,197,0,466,635.96,1,1,19,50,0.380000,0.096447,date
2,2026-02-01,1971,197,0,444,521.46,1,1,21,52,0.403846,0.106599,date
3,2026-03-01,1969,196,0,449,675.00,4,2,22,54,0.407407,0.112245,date
4,2026-04-01,1965,196,0,468,675.00,4,2,23,56,0.410714,0.117347,date
5,2026-05-01,1959,195,0,450,583.58,1,1,26,58,0.448276,0.133333,date
6,2026-06-01,1951,195,0,492,610.59,1,1,23,60,0.383333,0.117949,date
7,2026-07-01,1941,194,0,437,544.14,1,1,25,62,0.403226,0.128866,date
8,pooled,15696,1566,<NA>,<NA>,NaN,<NA>,<NA>,175,440,0.397727,0.111750,average
9,mean_of_dates,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,0.396267,0.111802,average


Тот же результат вторым путём: сортировка кортежей в чистом Python, без pandas. Совпадение числителей на каждой дате подтверждает, что правило и K воспроизводятся.

In [23]:
recount = []
for snapshot_at, day in train_rows.groupby("snapshot_at"):
    rows = [(r.debt_prev_rub != r.debt_prev_rub, -(r.debt_prev_rub if r.debt_prev_rub == r.debt_prev_rub else 0),
             r.customer_id, int(r.churn_target)) for r in day.itertuples()]
    k = len(rows) * CAMPAIGN_FRACTION.numerator // CAMPAIGN_FRACTION.denominator  # integer floor
    recount.append(sum(t[3] for t in sorted(rows)[:k]))
CHARTER_K = {"2025-12-01": 196, "2026-01-01": 197, "2026-02-01": 197, "2026-03-01": 196, "2026-04-01": 196,
             "2026-05-01": 195, "2026-06-01": 195, "2026-07-01": 194}  # project_charter.md, section 5
record_check("baseline TP@K and K reproduce with an independent pure-Python ranking",
             recount == baseline.tp_at_k.tolist() and baseline.k.tolist() == [CHARTER_K[d] for d in baseline.snapshot_at],
             f"TP@K per date {recount}; K = floor(0.1 * N) matches the charter table")

*Шаг 8. Базовое правило на train*

---

## Шаг 9. Воспроизводимость

1. Все таблицы строятся заново из входных файлов и сравниваются с уже построенными.
2. Контрольные суммы всех файлов, которые пишет ноутбук, сравниваются с предыдущим запуском (`output/checksums.csv`). При первом запуске сравнивать не с чем. Если код или данные менялись намеренно, проверка один раз остановит ноутбук со списком изменившихся файлов; следующий запуск сравнит уже с новой версией.

In [24]:
rebuilt_features, rebuilt_audit = build_all(SOURCES)
rebuilt_labels = pd.concat([churn_target(active_base(lifecycle, T), lifecycle, T) for T in ALL_DATES], ignore_index=True)
X_again = preprocessing_template.make_preprocessor(NUMERIC_X, CATEGORICAL_X).fit(inputs["train"]).transform(inputs["validation"])
record_check("rebuilding features, audit, labels and the transform gives identical results",
             bool(rebuilt_features.equals(features) and rebuilt_audit.equals(feature_audit)
                  and rebuilt_labels.equals(labels) and np.array_equal(X_again, X["validation"])),
             f"features {features.shape}, audit {feature_audit.shape}, labels {labels.shape}")

checks = pd.DataFrame(CHECKS)
write_csv(checks, OUTPUT / "checks.csv")

generated = sorted(p for p in OUTPUT.rglob("*") if p.is_file() and p.name not in {"checks.csv", "checksums.csv"})
generated.append(DOCS / "feature_catalog.csv")
current = pd.DataFrame({"file": [p.as_posix() for p in generated],
                        "sha256": [hashlib.sha256(p.read_bytes()).hexdigest() for p in generated]})
checksums_path = OUTPUT / "checksums.csv"
if checksums_path.exists():
    previous = read_csv(checksums_path)
    merged = current.merge(previous, on="file", how="outer", suffixes=("", "_previous"), indicator=True)
    changed = merged[(merged._merge != "both") | (merged.sha256 != merged.sha256_previous)].file.tolist()
    write_csv(current, checksums_path)
    record_check("all written files are byte-identical to the previous run", not changed,
                 f"{len(current)} files; changed: {changed or 'none'}")
else:
    write_csv(current, checksums_path)
    record_check("all written files are byte-identical to the previous run", True,
                 f"first run: {len(current)} checksums saved, nothing to compare")

checks = pd.DataFrame(CHECKS)
write_csv(checks, OUTPUT / "checks.csv")
print(f"{int(checks.passed.sum())} of {len(checks)} checks passed")
checks

28 of 28 checks passed


,check,passed,detail
0,"digital: key (customer_id, usage_month) is unique",True,"24098 rows, duplicates: 0"
1,digital: every customer_id is in subscribers_c...,True,unknown ids: 0
2,"digital: (customer, month) pairs equal the bil...",True,no new customers and no new months
3,"digital: observation_status in {observed, not_...",True,"{'observed': 16991, 'not_enabled': 6120, 'expo..."
4,"digital: usage_month is a first day, Sep 2025 ...",True,2025-09 … 2026-08
5,digital: available_at is not earlier than the ...,True,"delay after month end, hours: [0.0, 6.0, 24.0,..."
6,"digital: observed rows have all counts, 1 <= o...",True,16991 observed rows
7,digital: not_enabled and export_gap rows have ...,True,7107 rows
8,digital: observed_days equals billing active_d...,True,normalisation by observed_days covers exactly ...
9,"key (customer_id, snapshot_at) is unique in fe...",True,duplicates: 0; all four layers share keys and ...


*Шаг 9. Воспроизводимость*

---